# Importare dati di fMRI
In questa sezione prenderemo familiarità con il dato di risonanza funzionale.

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import mne
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from nilearn import plotting
from nilearn.signal import clean

In [ ]:
# we analyze the rest recording here
fmri_file = 'sub16ses2/func/sub-16_ses-02_task-rest_bold/func_minimal/func_mc.nii.gz' # the mc here stands for "Motion corrected"

# Atlas in the space of the recording
atlas_file = 'sub16ses2/func/sub-16_ses-02_task-rest_bold/func_seg/aparc+aseg2func.nii.gz'

lut_file = './FreeSurferColorLUT.txt'

In [ ]:
# let's start by opening the recording and printing some attributes

# load the image
fmri_img = nib.load(fmri_file)

print(f'The shape of the volume is {fmri_img.shape[:3]}')
print(f"The voxel dimensions are ({float(fmri_img.header.get_zooms()[0]):.2f}, {float(fmri_img.header.get_zooms()[1]):.2f}, {float(fmri_img.header.get_zooms()[2]):.2f})", fmri_img.header.get_xyzt_units()[0])

print(f'There are {fmri_img.shape[-1]} time points in this recording')
print('The repetition time is', fmri_img.header.get_zooms()[3], fmri_img.header.get_xyzt_units()[1])

In [ ]:
# store the repetition time
fmri_tr = fmri_img.header.get_zooms()[3]

print(f'The total length of the recording is {fmri_tr*fmri_img.shape[-1]:.0f} s.')

In [ ]:
# Lets plot a couple of time series to get an idea

fmri_data = fmri_img.get_fdata()
# Pick some specific voxels (X, Y, Z) to look at!
# Let's find the exact center of the grid 
# (This usually lands somewhere in the middle of the brain)
mid_x = fmri_data.shape[0] // 2
mid_y = fmri_data.shape[1] // 2
mid_z = fmri_data.shape[2] // 2

# Voxel 1: The dead center of the brain
voxel_center1 = fmri_data[mid_x, mid_y, mid_z, :]

# Voxel 2: Just above Voxel 2
voxel_center2 = fmri_data[mid_x, mid_y, mid_z + 1, :]

# Voxel 3: Let's pick a voxel near the edge of the grid
# (This is likely outside the brain)
voxel_edge = fmri_data[10, 10, mid_z, :]

# Plot the raw voxel time series
plt.figure(figsize=(12, 5))

# Plot the center voxels
plt.plot(voxel_center1, label=f'Center Voxel 1 ({mid_x}, {mid_y}, {mid_z})', color='blue', linewidth=2)
plt.plot(voxel_center2, label=f'Center Voxel 2 ({mid_x}, {mid_y}, {mid_z + 1})', color='green', linewidth=2)

# Plot the edge/air voxel
plt.plot(voxel_edge, label=f'Edge Voxel (10, 10, {mid_z})', color='red', alpha=0.7)

plt.title('Raw Single-Voxel fMRI Time Series', fontsize=16)
plt.xlabel('Time (TRs)', fontsize=14)
plt.ylabel('fMRI Signal', fontsize=14)
plt.legend(loc='best', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()

In [ ]:
# Let's compute the average value of each voxel and make an histogram
mean_volume = np.mean(fmri_data, axis=-1)

# Flatten the 3D grid into a single 1D list of numbers for the histogram
flat_data = mean_volume.flatten()

# Filter out the Background (all 0s)
# The air outside the head usually has a raw signal near 0. 
# We'll only keep voxels that have a meaningful signal (the threshold is arbitrary here)
brain_voxels = flat_data[flat_data > 100]

# Plot the Histogram
plt.figure(figsize=(10, 6))

# We use 100 'bins' to get a really detailed curve
plt.hist(brain_voxels, bins=100, color='mediumpurple', edgecolor='black', alpha=0.7)

plt.title(f'Histogram of Average Voxel Intensities ({len(brain_voxels)} total voxels)', fontsize=16)
plt.xlabel('fMRI Signal', fontsize=14)
plt.ylabel('Number of Voxels (Count)', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

plt.show()

# Aggregare le serie temporali
Lavorare con le serie temporali di ogni voxel nel volume può essere proibitivo (per motivi computazionali e interpretativi), se assumiamo che una singola regione cerebrale sia tutta attiva in un certo momento (che può non essere un'assunzione così sbagliata considerando come vengono creati gli atlanti), allora possiamo aggregare (i.e. mediare sull'intera regione) le serie temporali corrispondenti a una singola regione e ridurre drasticamente la dimensionalità del nostro dato.

In [ ]:
# let's start by loading the atlas and filtering out uninteresting regions

# The atlas image contains hundreds of integer IDs. Nilearn will extract
# time series for every ID present, in numerical order. We need to figure
# out exactly which IDs are in THIS specific brain.
atlas_img = nib.load(atlas_file)
atlas_data = atlas_img.get_fdata()

# let's filter out all regions that are not in the cortex
# this is uncommon in fmri, as it easily reaches subcortical regions
# we stick to the cortex here for simplicity
atlas_data[atlas_data<1000] = 0
atlas_data[atlas_data>=3000] = 0

# overwrite the atlas with a new atlas that only contains cortical values
atlas_img = nib.Nifti1Image(atlas_data, atlas_img.affine, atlas_img.header)

# plot the new atlas
plotting.plot_anat(atlas_img, title='Filtered Atlas', cmap = 'prism')

plotting.show()
# the background is red because the colormap we chose isn't black at 0

In [ ]:
# let's associate a name to each of these regions

# names of each region
fsLUT = mne.read_freesurfer_lut('./FreeSurferColorLUT.txt')[0]

unique_ids = np.unique(atlas_img.get_fdata()).astype(int)

# Remove 0 (Background / Unknown)
unique_ids = unique_ids[unique_ids != 0]

# Build our list of plain-English names in the exact order Nilearn expects
region_names = []

for name, label in fsLUT.items():
    if label in unique_ids:
        region_names.append(name)

In [ ]:
# now we aggregate the time series on a per region basis
# aggregation is done by averaging the time series on the voxels that belong to the same region

# the options we set are preprocessing steps:
# detrend=True: MRI machines are very sensitive, and the signal can be altered even by things like temperature
#               during the recording the machine generates heat, so the signal will slowly drift,
#               detrend fixes this be fitting and subtracting a line to the signals
# low_pass=0.1 : cuts out high-frequency noise (anything faster than 0.1 cycles per second), which removes the subject's heartbeat and respiration
# high_pass=0.01: cuts out ultra-slow noise, like persistent scanner artifacts
# standardize='zscore_sample':  Different parts of the brain and different tissue types have totally different raw magnetic baseline values.
#                               Standardizing forces the time series of every region to have a mean of 0 and a variance of 1.
#                               This ensures you are comparing the percent change in neural activity fairly across all regions,
#                                  rather than comparing raw water density
masker = NiftiLabelsMasker(labels_img=atlas_img, standardize='zscore_sample', verbose=1, detrend=True, low_pass=0.1, high_pass=0.01, t_r=fmri_tr)

# the previous step is used to set the options, this one triggers the computations and creates the output
time_series = masker.fit_transform(fmri_file)

In [ ]:
# Let's visualize a couple of these time series now

# ==========================================================
# 1. Define our two pairs of regions
# ==========================================================
# Pair 1: Expected to be highly synchronized (Mirror / Homotopic)
p1_regionA = 'ctx-lh-precentral'        # Left motor
p1_regionB = 'ctx-rh-precentral'        # Right motor

# Pair 2: Expected to be uncorrelated (DMN vs Somatosensory)
p2_regionA = 'ctx-lh-posteriorcingulate' # DMN Hub
p2_regionB = 'ctx-lh-postcentral'        # Somatosensory

# ==========================================================
# 2. Find their exact column indices
# ==========================================================
p1A_idx = region_names.index(p1_regionA)
p1B_idx = region_names.index(p1_regionB)

p2A_idx = region_names.index(p2_regionA)
p2B_idx = region_names.index(p2_regionB)

# ==========================================================
# 3. Create a figure with 2 rows and 1 column of plots
# ==========================================================
# sharex=True locks the time axis so zooming panning works on both together!
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 8), sharex=True)

# --- Top Plot: The Motor Network ---
axes[0].plot(time_series[:, p1A_idx], label='Left Motor', color='dodgerblue', linewidth=2)
axes[0].plot(time_series[:, p1B_idx], label='Right Motor', color='darkorange', linewidth=2)
axes[0].set_title('Highly Synchronized (Homotopic Motor Areas)', fontsize=14)
axes[0].set_ylabel('Standardized BOLD')
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle='--', alpha=0.6)

# --- Bottom Plot: DMN vs Somatosensory ---
axes[1].plot(time_series[:, p2A_idx], label='DMN (Post. Cingulate)', color='mediumseagreen', linewidth=2)
axes[1].plot(time_series[:, p2B_idx], label='Somatosensory (Postcentral)', color='crimson', linewidth=2)
axes[1].set_title('Unrelated Networks (Global Signal / Noise Present)', fontsize=14)
axes[1].set_xlabel('Time (Scanner Volumes / TRs)', fontsize=12)
axes[1].set_ylabel('Standardized BOLD')
axes[1].legend(loc='upper right')
axes[1].grid(True, linestyle='--', alpha=0.6)

# Clean up the layout and show
plt.tight_layout()
plt.show()

### Il Segnale Globale (Global Signal) nell'fMRI
I plot di sopra sembrano *molto* correlati, questo è perché hanno una grossa componente di cosiddetto Segnale Globale. Questo fenomeno può mascherare l'attività neurale effettiva.

Il segnale BOLD utilizzato dalla risonanza magnetica funzionale (fMRI) misura l'ossigenazione del sangue. Durante una scansione, intervengono fattori fisiologici che influenzano l'intero cervello simultaneamente:
- Fluttuazioni della respirazione.
- Variazioni della frequenza cardiaca.
- Grandi ondate di sangue che irrorano il sistema circolatorio cerebrale.

Poiché regioni diverse appartengono allo stesso sistema circolatorio, "cavalcano" la stessa onda fisiologica. Questo spinge artificialmente la loro correlazione verso valori positivi, nascondendo il fatto che i loro impulsi neurali siano in realtà indipendenti.

Per isolare la vera attività neurale, i neuroscienziati utilizzano una tecnica chiamata Global Signal Regression (GSR):
- **Calcolo**: Si ottiene la media del segnale di tutto il cervello per ogni istante temporale.
- **Sottrazione**: Si sottrae matematicamente questo segnale comune (riscalato opportunamente) da ogni singola regione.
- **Risultato**: Una volta rimosso il rumore globale, le diverse reti cerebrali tornano a mostrare la loro indipendenza (correlazione zero o negativa).

In [ ]:
# Calculate the Global Signal (the mean across all regions for each TR)
global_signal = np.mean(time_series, axis=1, keepdims = True)

# Clean the time series by regressing out the global signal
cleaned_time_series = clean(time_series, confounds=global_signal, standardize='zscore_sample')

In [ ]:
# Let's visualize a couple of these time series now

# ==========================================================
# 1. Define our two pairs of regions
# ==========================================================
# Pair 1: Expected to be highly synchronized (Mirror / Homotopic)
p1_regionA = 'ctx-lh-precentral'        # Left motor
p1_regionB = 'ctx-rh-precentral'        # Right motor

# Pair 2: Expected to be uncorrelated (DMN vs Somatosensory)
p2_regionA = 'ctx-lh-posteriorcingulate' # DMN Hub
p2_regionB = 'ctx-lh-postcentral'        # Somatosensory

# ==========================================================
# 2. Find their exact column indices
# ==========================================================
p1A_idx = region_names.index(p1_regionA)
p1B_idx = region_names.index(p1_regionB)

p2A_idx = region_names.index(p2_regionA)
p2B_idx = region_names.index(p2_regionB)

# ==========================================================
# 3. Create a figure with 2 rows and 1 column of plots
# ==========================================================
# sharex=True locks the time axis so zooming panning works on both together!
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 8), sharex=True)

# --- Top Plot: The Motor Network ---
axes[0].plot(cleaned_time_series[:, p1A_idx], label='Left Motor', color='dodgerblue', linewidth=2)
axes[0].plot(cleaned_time_series[:, p1B_idx], label='Right Motor', color='darkorange', linewidth=2)
axes[0].set_title('Highly Synchronized (Homotopic Motor Areas)', fontsize=14)
axes[0].set_ylabel('Standardized BOLD')
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle='--', alpha=0.6)

# --- Bottom Plot: DMN vs Somatosensory ---
axes[1].plot(cleaned_time_series[:, p2A_idx], label='DMN (Post. Cingulate)', color='mediumseagreen', linewidth=2)
axes[1].plot(cleaned_time_series[:, p2B_idx], label='Somatosensory (Postcentral)', color='crimson', linewidth=2)
axes[1].set_title('Unrelated Networks (Global Signal / Noise Present)', fontsize=14)
axes[1].set_xlabel('Time (Scanner Volumes / TRs)', fontsize=12)
axes[1].set_ylabel('Standardized BOLD')
axes[1].legend(loc='upper right')
axes[1].grid(True, linestyle='--', alpha=0.6)

# Clean up the layout and show
plt.tight_layout()
plt.show()

# Estrarre la matrice di connettività funzionale
Ora vedremo come estrarre la cosiddetta connettività funzionale dalla risonanza. Visto che stiamo analizzando una registrazione a riposo, la connettività che ricaveremo è il cosiddetto **Default Mode Network**, una rete molto studiata in letteratura.

In [ ]:
# Compute the Connectivity Matrix by calculating the Pearson correlation between all pairs of regions
correlation_measure = ConnectivityMeasure(kind='correlation', standardize = 'zscore_sample')
correlation_matrix = correlation_measure.fit_transform(cleaned_time_series)[0]
correlation_matrix

In [ ]:
# plot this matrix with labels on the side

plotting.plot_matrix(
    correlation_matrix,
    labels=region_names,
    figure=(12, 12),
    vmax=1.0, 
    vmin=-1.0,
    colorbar=True,
    title='Functional Connectivity Matrix'
)

# Make the font smaller so it doesn't overlap
plt.xticks(fontsize=6)
plt.yticks(fontsize=6)

plt.show()